<a href="https://colab.research.google.com/github/milvus-io/bootcamp/blob/master/integration/build_RAG_with_milvus_and_exa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>   <a href="https://github.com/milvus-io/bootcamp/blob/master/integration/build_RAG_with_milvus_and_exa.ipynb" target="_blank">
    <img src="https://img.shields.io/badge/View%20on%20GitHub-555555?style=flat&logo=github&logoColor=white" alt="GitHub Repository"/>
</a>

# Building a Dual-Source RAG Agent with Exa and Milvus

This tutorial demonstrates how to build an agent that searches both **the public web** (via [Exa](https://exa.ai/)) and **a private knowledge base** (via [Milvus](https://milvus.io/)), then synthesizes a unified answer. The agent uses OpenAI's function calling to automatically decide which source to query based on the user's question.

[Exa](https://exa.ai/) is a search API designed for AI applications. Unlike traditional keyword-based search engines, Exa supports semantic (neural) search — you describe what you want in natural language and it understands your intent. It also provides content extraction, highlights, and category-based filtering. [Milvus](https://milvus.io/) is an open-source vector database built for scalable similarity search. By combining them with an LLM agent, you can build a system that retrieves both internal proprietary data and up-to-date web information in a single workflow.

## Prerequisites

Before running this notebook, make sure you have the following dependencies installed:

In [ ]:
! pip install exa_py pymilvus openai

> If you are using Google Colab, to enable dependencies just installed, you may need to **restart the runtime** (click on the "Runtime" menu at the top of the screen, and select "Restart session" from the dropdown menu).

You will need API keys from [Exa](https://dashboard.exa.ai/api-keys) and [OpenAI](https://platform.openai.com/api-keys). Set them as environment variables:

In [ ]:
import os

os.environ["EXA_API_KEY"] = "***********"
os.environ["OPENAI_API_KEY"] = "sk-***********"

## Initialize Clients

Set up the Exa, OpenAI, and Milvus clients. We use OpenAI's `text-embedding-3-small` model to generate vector embeddings, and Milvus Lite for local vector storage with zero infrastructure setup.

In [3]:
import json
from openai import OpenAI
from pymilvus import MilvusClient, DataType
from exa_py import Exa

llm = OpenAI()
exa = Exa(api_key=os.environ["EXA_API_KEY"])
milvus = MilvusClient(uri="./milvus_exa_demo.db")

EMBED_MODEL = "text-embedding-3-small"
EMBED_DIM = 1536
COLLECTION = "private_kb"

> As for the argument of `MilvusClient`:
> - Setting the `uri` as a local file, e.g.`./milvus.db`, is the most convenient method, as it automatically utilizes [Milvus Lite](https://milvus.io/docs/milvus_lite.md) to store all data in this file.
> - If you have large scale of data, you can set up a more performant Milvus server on [docker or kubernetes](https://milvus.io/docs/quickstart.md). In this setup, please use the server uri, e.g.`http://localhost:19530`, as your `uri`.
> - If you want to use [Zilliz Cloud](https://zilliz.com/cloud), the fully managed cloud service for Milvus, adjust the `uri` and `token`, which correspond to the [Public Endpoint and Api key](https://docs.zilliz.com/docs/on-zilliz-cloud-console#free-cluster-details) in Zilliz Cloud.

Define a helper function to generate embeddings. We will reuse this across the notebook for both indexing and querying:

In [4]:
def embed_text(text: str | list[str]) -> list:
    """Generate embedding vector(s) using OpenAI."""
    resp = llm.embeddings.create(
        input=text if isinstance(text, list) else [text],
        model=EMBED_MODEL,
    )
    if isinstance(text, list):
        return [item.embedding for item in resp.data]
    return resp.data[0].embedding

## Build the Private Knowledge Base (Milvus)

We simulate a set of internal company documents — product specs, policies, earnings reports, and API docs — that would not appear on the public web. In a real scenario, these could come from your internal wikis, databases, or document management systems.

In [5]:
private_docs = [
    {
        "id": 1,
        "text": (
            "Acme Widget Pro supports up to 10,000 concurrent connections. "
            "It uses a proprietary compression algorithm (AcmeZip v3) that "
            "reduces payload size by 72% compared to gzip."
        ),
        "source": "product-spec.pdf",
    },
    {
        "id": 2,
        "text": (
            "Our return policy allows customers to return any product within "
            "30 days of purchase for a full refund. After 30 days, only store "
            "credit is offered. Damaged items must be reported within 48 hours."
        ),
        "source": "return-policy.md",
    },
    {
        "id": 3,
        "text": (
            "Q3 2025 revenue was $4.2M, up 18% from Q2. The growth was "
            "primarily driven by enterprise customers adopting Widget Pro. "
            "Churn rate dropped to 3.1%."
        ),
        "source": "q3-earnings.pdf",
    },
    {
        "id": 4,
        "text": (
            "Internal API rate limits: free tier 100 req/min, pro tier "
            "5,000 req/min, enterprise tier 50,000 req/min. Rate limit "
            "headers are X-RateLimit-Remaining and X-RateLimit-Reset."
        ),
        "source": "api-docs.md",
    },
    {
        "id": 5,
        "text": (
            "Employee onboarding checklist: 1) Sign NDA, 2) Set up VPN access, "
            "3) Enroll in mandatory security training, 4) Request Jira and "
            "Confluence access from IT, 5) Schedule 1:1 with manager."
        ),
        "source": "onboarding-guide.md",
    },
]

Create the Milvus collection with an explicit schema, embed the documents, and insert them:

In [ ]:
if milvus.has_collection(COLLECTION):
    milvus.drop_collection(COLLECTION)

schema = milvus.create_schema(auto_id=False, enable_dynamic_field=True)
schema.add_field(field_name="id", datatype=DataType.INT64, is_primary=True)
schema.add_field(field_name="vector", datatype=DataType.FLOAT_VECTOR, dim=EMBED_DIM)
schema.add_field(field_name="text", datatype=DataType.VARCHAR, max_length=65535)
schema.add_field(field_name="source", datatype=DataType.VARCHAR, max_length=512)

index_params = milvus.prepare_index_params()
index_params.add_index(
    field_name="vector", index_type="AUTOINDEX", metric_type="COSINE"
)

milvus.create_collection(
    collection_name=COLLECTION,
    schema=schema,
    index_params=index_params,
    # consistency_level="Strong",
)

# Embed all documents in one batch call
embeddings = embed_text([doc["text"] for doc in private_docs])

milvus.insert(
    collection_name=COLLECTION,
    data=[
        {
            "id": doc["id"],
            "vector": emb,
            "text": doc["text"],
            "source": doc["source"],
        }
        for doc, emb in zip(private_docs, embeddings)
    ],
)

print(f"Inserted {len(private_docs)} documents into Milvus.")

Inserted 5 documents into Milvus.


Let's verify the retrieval works with a quick test query:

In [7]:
query = "What is the return policy?"
results = milvus.search(
    collection_name=COLLECTION,
    data=[embed_text(query)],
    limit=2,
    output_fields=["text", "source"],
)

for hit in results[0]:
    print(f"[score={hit['distance']:.3f}] ({hit['entity']['source']})")
    print(f"  {hit['entity']['text'][:120]}...")
    print()

[score=0.665] (return-policy.md)
  Our return policy allows customers to return any product within 30 days of purchase for a full refund. After 30 days, on...

[score=0.119] (q3-earnings.pdf)
  Q3 2025 revenue was $4.2M, up 18% from Q2. The growth was primarily driven by enterprise customers adopting Widget Pro. ...



## Explore Exa Search Capabilities

Before building the agent, let's explore Exa's search features. Exa supports multiple search modes that are useful for different scenarios.

**Semantic search** with content extraction — Exa can return not only links but also the article text, key highlights, and AI-generated summaries in a single request:

In [8]:
web_results = exa.search_and_contents(
    query="latest trends in AI agents 2025",
    type="auto",
    num_results=3,
    text={"max_characters": 3000},
    highlights={"num_sentences": 3},
)

for r in web_results.results:
    print(f"[{r.title}]")
    print(f"  URL: {r.url}")
    if r.highlights:
        print(f"  Highlight: {r.highlights[0][:150]}...")
    print()

[AI Agent Trends of 2025: Entering the Agentic Era of Autonomous Intelligence]
  URL: https://genesishumanexperience.com/2025/10/19/ai-agent-trends-of-2025-entering-the-agentic-era-of-autonomous-intelligence/
  Highlight:  Agent Trends of 2025: Entering the Agentic Era of AutonomousIntelligence
By Dr Luke Soon 1. The Agentic Shift — From Generative AI to Autonomous Coll...

[AI Agent Technology Trends 2025: Tools, Frameworks, and What’s Next | The AI Journal]
  URL: https://aijourn.com/ai-agent-technology-trends-2025-tools-frameworks-and-whats-next/
  Highlight:  Technology Trends 2025: Tools, Frameworks, and What’s Next
## By Sergii Opanasenko, Co-founder, Greenice  ...    AI agents are rapidly moving from pr...

[AI Agents Revolutionized B2B Marketing in 2025: From Automation ...]
  URL: https://www.demandgenreport.com/industry-news/feature/ai-agents-revolutionize-b2b-marketing-in-2025-from-automation-to-strategy/51106/
  Highlight: �
Trending  ...    Typeface New Offering
2026 B2B T

**Category-based filtering** — you can restrict results to specific content types such as `"research paper"`, `"news"`, `"company"`, or `"tweet"`. This is useful when you want high-quality sources and want to avoid noise:

In [9]:
filtered_results = exa.search_and_contents(
    query="vector database performance benchmarks",
    category="research paper",
    num_results=3,
    highlights={"num_sentences": 2},
)

for r in filtered_results.results:
    print(f"- {r.title}")
    print(f"  {r.url}\n")

- Vector Database Benchmark 2026 | Top 10 Compared - Salt Technologies AI
  https://www.salttechno.ai/datasets/vector-database-performance-benchmark-2026/

- Benchmarking Vector Databases
  https://qdrant.tech/benchmarks/

- VectorDBBench: An Open-Source VectorDB Benchmark Tool
  https://zilliz.com/vdbbench-leaderboard



**Find similar articles** — given a URL, Exa can find other articles with similar content. This is helpful for expanding research from a good starting point:

In [10]:
if web_results.results:
    source_url = web_results.results[0].url
    similar = exa.find_similar_and_contents(
        url=source_url,
        num_results=3,
        highlights={"num_sentences": 2},
    )
    print(f"Articles similar to: {source_url}\n")
    for r in similar.results:
        print(f"- {r.title}")
        print(f"  {r.url}\n")

Articles similar to: https://genesishumanexperience.com/2025/10/19/ai-agent-trends-of-2025-entering-the-agentic-era-of-autonomous-intelligence/

- The Six Types of AI Agents in 2025 – Genesis: Human Experience in the Age of Artificial Intelligence | Synthesis: The Superintelligence Protocol
  https://genesishumanexperience.com/2025/09/28/the-six-types-of-ai-agents-you-should-know-in-2025/

- Agentic AI Concepts: Architecting Autonomous Intelligence – Genesis: Human Experience in the Age of Artificial Intelligence | Synthesis: The Superintelligence Protocol
  https://genesishumanexperience.com/2025/10/01/agentic-ai-concepts-architecting-the-next-frontier-of-autonomous-intelligence/



## Define the Agent Tools

Now we define the two tool functions that the agent will use. The private KB tool searches Milvus using vector similarity, while the web tool searches the public internet via Exa:

In [11]:
def search_private_kb(query: str) -> str:
    """Search the internal knowledge base using Milvus vector search."""
    results = milvus.search(
        collection_name=COLLECTION,
        data=[embed_text(query)],
        limit=3,
        output_fields=["text", "source"],
    )
    chunks = []
    for hit in results[0]:
        chunks.append(f"[{hit['entity']['source']}] {hit['entity']['text']}")
    return "\n\n".join(chunks) if chunks else "No relevant internal documents found."


def search_web(query: str) -> str:
    """Search the public web using Exa for up-to-date information."""
    results = exa.search_and_contents(
        query=query,
        type="auto",
        num_results=3,
        highlights={"num_sentences": 3},
    )
    items = []
    for r in results.results:
        highlight = r.highlights[0] if r.highlights else "No snippet available."
        items.append(f"[{r.title}]({r.url})\n{highlight}")
    return "\n\n".join(items) if items else "No web results found."


TOOL_FNS = {
    "search_private_kb": search_private_kb,
    "search_web": search_web,
}

## Build the Agent

The agent uses OpenAI's [function calling](https://platform.openai.com/docs/guides/function-calling) to decide which tool(s) to invoke. It follows a simple loop: the LLM receives the user query, decides which tools to call (if any), executes them, and then synthesizes a final answer from the retrieved context.

In [12]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "search_private_kb",
            "description": (
                "Search the company's internal knowledge base (product docs, "
                "policies, earnings, API docs, HR guides). Use this for any "
                "question about internal/proprietary information."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "The search query"}
                },
                "required": ["query"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "search_web",
            "description": (
                "Search the public web for up-to-date external information - "
                "news, trends, competitor analysis, open-source projects, etc. "
                "Use this when the question is about the outside world."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "The search query"}
                },
                "required": ["query"],
            },
        },
    },
]

SYSTEM_PROMPT = """You are a helpful assistant with access to two search tools:

1. **search_private_kb** - searches the company's internal knowledge base.
2. **search_web** - searches the public internet via Exa.

Routing rules:
- Questions about internal products, policies, metrics, or processes: use search_private_kb.
- Questions about external trends, news, competitors, or general knowledge: use search_web.
- Questions that need both internal and external context: call BOTH tools, then synthesize.

Always cite your sources. For internal docs, mention the filename. For web results, include the URL."""


def run_agent(user_query: str) -> str:
    """Run the agent loop: LLM -> tool calls -> LLM -> final answer."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_query},
    ]

    print(f"User: {user_query}\n")

    # First LLM call - may request tool calls
    response = llm.chat.completions.create(
        model="gpt-4o",
        messages=messages,
        tools=TOOLS,
    )
    msg = response.choices[0].message
    messages.append(msg)

    # If no tool calls, return directly
    if not msg.tool_calls:
        print(f"Agent (no tools used): {msg.content}")
        return msg.content

    # Execute each tool call
    for tc in msg.tool_calls:
        fn_name = tc.function.name
        fn_args = json.loads(tc.function.arguments)
        print(f"  -> Calling {fn_name}(query={fn_args['query']!r})")

        result = TOOL_FNS[fn_name](**fn_args)
        messages.append(
            {
                "role": "tool",
                "tool_call_id": tc.id,
                "content": result,
            }
        )

    # Second LLM call - synthesize final answer
    response = llm.chat.completions.create(
        model="gpt-4o",
        messages=messages,
        tools=TOOLS,
    )
    answer = response.choices[0].message.content
    print(f"\nAgent:\n{answer}")
    return answer

## Demo

Now let's test the agent with three scenarios that demonstrate different routing behaviors.

### Scenario A: Internal question (routes to Milvus)

Ask about an internal policy — the agent should call `search_private_kb` and retrieve the answer from our private documents:

In [13]:
run_agent("What is the return policy for Acme products?")

User: What is the return policy for Acme products?

  -> Calling search_private_kb(query='Acme products return policy')

Agent:
Acme's return policy allows customers to return any product within 30 days of purchase for a full refund. After 30 days, returns are eligible for store credit only. Additionally, any damaged items must be reported within 48 hours of receipt (source: return-policy.md).


"Acme's return policy allows customers to return any product within 30 days of purchase for a full refund. After 30 days, returns are eligible for store credit only. Additionally, any damaged items must be reported within 48 hours of receipt (source: return-policy.md)."

### Scenario B: External question (routes to Exa)

Ask about external trends — the agent should call `search_web` to fetch up-to-date information from the public internet:

In [14]:
run_agent("What are the latest AI agent frameworks trending in 2025?")

User: What are the latest AI agent frameworks trending in 2025?

  -> Calling search_web(query='latest AI agent frameworks 2025')

Agent:
As of 2025, some trending AI agent frameworks include:

1. **LangChain**: Known for its role in building applications with large language models (LLMs), LangChain specializes in prompt chaining, tool usage, and memory management. It's particularly useful for creating LLM-powered applications ([source](https://odsc.medium.com/top-10-open-source-ai-agent-frameworks-to-know-in-2025-c739854ec859)).

2. **Langflow**: This framework supports visual-first development and is useful for agentic and retrieval-augmented generation (RAG) applications. It includes comprehensive features for web/file search, code execution, and integration with various tools ([source](https://www.langflow.org/blog/the-complete-guide-to-choosing-an-ai-agent-framework-in-2025)).

3. **Agent0**: A fully autonomous AI framework that develops high-performing agents through multi-step c

"As of 2025, some trending AI agent frameworks include:\n\n1. **LangChain**: Known for its role in building applications with large language models (LLMs), LangChain specializes in prompt chaining, tool usage, and memory management. It's particularly useful for creating LLM-powered applications ([source](https://odsc.medium.com/top-10-open-source-ai-agent-frameworks-to-know-in-2025-c739854ec859)).\n\n2. **Langflow**: This framework supports visual-first development and is useful for agentic and retrieval-augmented generation (RAG) applications. It includes comprehensive features for web/file search, code execution, and integration with various tools ([source](https://www.langflow.org/blog/the-complete-guide-to-choosing-an-ai-agent-framework-in-2025)).\n\n3. **Agent0**: A fully autonomous AI framework that develops high-performing agents through multi-step co-evolution without relying on external data. It integrates seamlessly with tools and emphasizes practical, data-free reinforcement

### Scenario C: Hybrid question (routes to both)

Ask a question that requires both internal specs and external benchmarks — the agent should call both tools and synthesize a comparison:

In [15]:
run_agent(
    "How does our Widget Pro's throughput compare to "
    "open-source alternatives on the market?"
)

User: How does our Widget Pro's throughput compare to open-source alternatives on the market?

  -> Calling search_private_kb(query='Widget Pro throughput vs open-source alternatives')
  -> Calling search_web(query='open-source alternatives to Widget Pro throughput comparison')

Agent:
Internally, the Widget Pro supports up to 10,000 concurrent connections and utilizes a proprietary compression algorithm (AcmeZip v3) that reduces payload size by 72% compared to gzip ([product-spec.pdf]). This robust throughput capability primarily positions Widget Pro favorably within enterprise settings ([q3-earnings.pdf]).

Externally, open-source alternatives like Elfsight and UserWay are often noted for lightweight review displays, fast load speeds, and user-friendly interfaces. Many of these alternatives also offer free plans with fewer capabilities and paid options for enhanced features. However, they may struggle with control and performance features compared to Widget Pro, which is engineered f

"Internally, the Widget Pro supports up to 10,000 concurrent connections and utilizes a proprietary compression algorithm (AcmeZip v3) that reduces payload size by 72% compared to gzip ([product-spec.pdf]). This robust throughput capability primarily positions Widget Pro favorably within enterprise settings ([q3-earnings.pdf]).\n\nExternally, open-source alternatives like Elfsight and UserWay are often noted for lightweight review displays, fast load speeds, and user-friendly interfaces. Many of these alternatives also offer free plans with fewer capabilities and paid options for enhanced features. However, they may struggle with control and performance features compared to Widget Pro, which is engineered for high concurrency and efficiency ([WiserNotify blog](https://wisernotify.com/blog/elfsight-alternatives/), [UserWay Comparison](https://userway.org/compare/)).\n\nIn summary, while open-source alternatives offer various functionalities and can be quite efficient in particular scena

## Cleanup

When you are done, drop the collection to free resources.

In [16]:
milvus.drop_collection(COLLECTION)

I0000 00:00:1773040980.114584 10485636 chttp2_transport.cc:1341] unix:/var/folders/wn/4wflyq8x0f9bhkwryvss30880000gn/T/tmpqscd3uj9_milvus_exa_demo.db.sock: Got goaway [11] err=UNAVAILABLE:GOAWAY received; Error code: 11; Debug Text: too_many_pings {http2_error:11, grpc_status:14}
E0000 00:00:1773040980.115112 10485636 chttp2_transport.cc:1370] unix:/var/folders/wn/4wflyq8x0f9bhkwryvss30880000gn/T/tmpqscd3uj9_milvus_exa_demo.db.sock: Received a GOAWAY with error code ENHANCE_YOUR_CALM and debug data equal to "too_many_pings". Current keepalive time (before throttling): 10000ms


## Conclusion

In this tutorial, we built a dual-source RAG agent that combines Milvus for private knowledge retrieval with Exa for public web search. The key components are:

- **Milvus** stores and retrieves internal documents via vector similarity search, ensuring proprietary data stays private and searchable.
- **Exa** provides semantic web search with features like category filtering, content extraction, and similar article discovery.
- **OpenAI function calling** enables the LLM to automatically route queries to the right source — or both — based on the question's intent.

This pattern is applicable to enterprise use cases where an AI assistant needs access to both confidential internal documents and real-time external information.